In [2]:
import warnings
warnings.filterwarnings("ignore")
 
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch, Circle, FancyArrowPatch
import matplotlib.patheffects as pe
import seaborn as sns
from math import radians, sin, cos, sqrt, atan2

### Định nghĩa các mã màu cho các biểu đồ


In [3]:
C = {
    "blue":   "#2563EB", "green":  "#10B981", "amber":  "#F59E0B",
    "red":    "#EF4444", "purple": "#8B5CF6", "teal":   "#06B6D4",
    "indigo": "#4F46E5", "rose":   "#F43F5E", "lime":   "#84CC16",
    "bg":     "#F8FAFC", "card":   "#FFFFFF", "title":  "#1E293B",
    "axis":   "#64748B", "grid":   "#E2E8F0",
}
PALETTE = [C["blue"], C["green"], C["amber"], C["red"],
           C["purple"], C["teal"], C["indigo"], C["rose"], C["lime"], C["axis"]]
 
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.facecolor":  C["bg"],
    "axes.facecolor":    C["card"],
    "axes.edgecolor":    C["grid"],
    "axes.labelcolor":   C["axis"],
    "xtick.color":       C["axis"],
    "ytick.color":       C["axis"],
    "grid.color":        C["grid"],
    "font.family":       "DejaVu Sans",
    "font.size":         9,
    "axes.titlesize":    11,
    "axes.titleweight":  "bold",
    "axes.titlecolor":   C["title"],
})

### Load dữ liệu và xử lí cho việc phân tích

In [4]:
df = pd.read_csv("../Data/fitness_cleaned.csv")
 
# Haversine
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2 - lat1); dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))
 
# Gym coordinates
gyms = df.groupby("home_gym_location")[["latitude","longitude"]].first().reset_index()
gyms.columns = ["city","lat","lon"]
gyms = gyms.set_index("city")
 
# Revenue / engagement per city
rev = df.groupby("home_gym_location").agg(
    members        = ("final_price","count"),
    total_revenue  = ("final_price","sum"),
    avg_price      = ("final_price","mean"),
    avg_visit      = ("visit_per_week","mean"),
    avg_duration   = ("duration_in_gym_minutes","mean"),
    pct_premium    = ("membership_type", lambda x: (x=="Premium").mean()*100),
    pct_elite      = ("membership_type", lambda x: (x=="Elite").mean()*100),
    pct_pt         = ("personal_training","mean"),
    pct_multi      = ("multi_location_access","mean"),
    revenue_pp     = ("final_price","mean"),   # revenue per person
).round(2)
 
rev = rev.join(gyms)
 
# Distance matrix between gyms
cities = gyms.index.tolist()
dist_mat = pd.DataFrame(index=cities, columns=cities, dtype=float)
for c1 in cities:
    for c2 in cities:
        dist_mat.loc[c1, c2] = haversine(
            gyms.loc[c1,"lat"], gyms.loc[c1,"lon"],
            gyms.loc[c2,"lat"], gyms.loc[c2,"lon"])
 
rev["nearest_km"] = [dist_mat.loc[c].drop(c).min() for c in rev.index]
rev["nearest_city"] = [dist_mat.loc[c].drop(c).idxmin() for c in rev.index]
 
# Growth score: higher avg_price + higher visit_rate + smaller existing base = more opportunity
rev["growth_score"] = (
    rev["avg_price"] / rev["avg_price"].max() * 0.35 +
    rev["avg_visit"] / rev["avg_visit"].max() * 0.30 +
    (1 - rev["members"] / rev["members"].max()) * 0.20 +
    rev["nearest_km"] / rev["nearest_km"].max() * 0.15   # isolation bonus
) * 100
 
# Short labels
SHORT = {
    "Anaheim, CA":       "Anaheim",
    "Bakersfield, CA":   "Bakersfield",
    "Fresno, CA":        "Fresno",
    "Long Beach, CA":    "Long Beach",
    "Los Angeles, CA":   "Los Angeles",
    "Oakland, CA":       "Oakland",
    "Sacramento, CA":    "Sacramento",
    "San Diego, CA":     "San Diego",
    "San Francisco, CA": "San Francisco",
    "San Jose, CA":      "San Jose",
}
rev["short"] = rev.index.map(SHORT)
 
print("Revenue overview:")
print(rev[["members","total_revenue","avg_price","avg_visit","nearest_km","growth_score"]].sort_values("total_revenue", ascending=False).to_string())
 

Revenue overview:
                   members  total_revenue  avg_price  avg_visit  nearest_km  growth_score
home_gym_location                                                                        
Fresno, CA             265        8983.30      33.90       2.61  166.707618     74.116994
Long Beach, CA         223        7402.32      33.19       2.58   26.854017     63.721843
San Diego, CA          201        7395.25      36.79       2.70  142.958407     80.469792
Sacramento, CA         226        7358.75      32.56       2.59  109.894493     70.478022
Bakersfield, CA        189        6841.82      36.20       2.54  163.085818     80.964102
San Francisco, CA      201        6798.15      33.82       2.65   13.429625     65.496869
Los Angeles, CA        173        6386.50      36.92       2.87   31.705621     74.465985
San Jose, CA           167        6205.78      37.16       2.80   61.951494     77.137158
Anaheim, CA            188        6023.20      32.04       2.88   26.854017     68

### GEO MAP + REVENUE BUBBLE + COVERAGE GAPS   

In [10]:
fig1 = plt.figure(figsize=(22, 18), facecolor=C["bg"])
fig1.suptitle("PHÂN TÍCH VỊ TRÍ GYM & DOANH THU — CALIFORNIA",
              fontsize=17, fontweight="bold", color=C["title"], y=0.98)
 
gs = gridspec.GridSpec(3, 3, figure=fig1, hspace=0.45, wspace=0.35,
                       left=0.05, right=0.97, top=0.93, bottom=0.04)
 
# ── 1A. Geo bubble map ────────────────────────────────────────────────────────
ax_map = fig1.add_subplot(gs[:2, :2])
ax_map.set_facecolor("#EFF6FF")
 
# Draw connections between nearby gyms (nearest neighbor)
drawn = set()
for c in cities:
    nc = rev.loc[c, "nearest_city"]
    pair = tuple(sorted([c, nc]))
    if pair not in drawn:
        x1, y1 = gyms.loc[c, "lon"],   gyms.loc[c, "lat"]
        x2, y2 = gyms.loc[nc, "lon"],  gyms.loc[nc, "lat"]
        d = dist_mat.loc[c, nc]
        lw = max(0.5, 3 - d/100)
        ax_map.plot([x1,x2],[y1,y2], color=C["grid"], lw=lw, zorder=1, alpha=0.7, linestyle="--")
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax_map.text(mx, my + 0.05, f"{d:.0f}km", fontsize=7, ha="center",
                    color=C["axis"], zorder=4)
        drawn.add(pair)
 
# Bubbles: size = total revenue, color = growth score
rev_sorted = rev.sort_values("total_revenue", ascending=False)
sizes = (rev["total_revenue"] / rev["total_revenue"].max() * 1800 + 200)
sc = ax_map.scatter(rev["lon"], rev["lat"],
                    s=sizes, c=rev["growth_score"],
                    cmap="RdYlGn", vmin=60, vmax=85,
                    alpha=0.85, zorder=3, edgecolors="white", linewidths=1.5)
 
# Gym labels
for city, row in rev.iterrows():
    ax_map.annotate(
        f"{row['short']}\n${row['total_revenue']:,.0f}",
        xy=(row["lon"], row["lat"]),
        xytext=(row["lon"] + 0.25, row["lat"] + 0.18),
        fontsize=8, fontweight="bold", color=C["title"], zorder=5,
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                  edgecolor=C["grid"], alpha=0.85, linewidth=0.8),
    )
 
# Proposed expansion zones (coverage gaps)
proposed = [
    # (lat, lon, label, reason)
    (33.45, -117.60, "★ Riverside\n(Đề xuất)", "Khoảng trống SD-Anaheim 143km"),
    (35.10, -119.40, "★ Santa Clarita\n(Đề xuất)", "Khoảng trống LA-Bakersfield"),
    (37.05, -120.85, "★ Modesto\n(Đề xuất)", "Khoảng trống Fresno-Sacramento 254km"),
]
for plat, plon, plabel, _ in proposed:
    ax_map.scatter(plon, plat, s=350, color=C["rose"], marker="*", zorder=6,
                   edgecolors="white", linewidths=0.8)
    ax_map.annotate(plabel, xy=(plon, plat), xytext=(plon - 0.9, plat - 0.3),
                    fontsize=8, color=C["rose"], fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.2", facecolor="#FFF1F2",
                              edgecolor=C["rose"], linewidth=1, alpha=0.9))
 
cbar = plt.colorbar(sc, ax=ax_map, shrink=0.6, pad=0.02)
cbar.set_label("Growth Score →", fontsize=9)
 
# Legend for bubble size
for rev_val, label in [(5000,"$5K"), (7000,"$7K"), (9000,"$9K")]:
    sz = rev_val / rev["total_revenue"].max() * 1800 + 200
    ax_map.scatter([], [], s=sz, color=C["axis"], alpha=0.4, label=f"Revenue {label}")
ax_map.legend(title="Bubble = Total Revenue", fontsize=8, loc="lower left",
              framealpha=0.85, title_fontsize=8)
 
ax_map.set_title("Bản đồ Gym — Doanh thu & Growth Score (★ = Vị trí đề xuất mở mới)",
                 fontsize=12)
ax_map.set_xlabel("Kinh độ (Longitude)"); ax_map.set_ylabel("Vĩ độ (Latitude)")
ax_map.set_xlim(-123.5, -116.0); ax_map.set_ylim(32.0, 39.3)
ax_map.grid(True, alpha=0.3)
 
# ── 1B. Total Revenue ranking ─────────────────────────────────────────────────
ax_rev = fig1.add_subplot(gs[0, 2])
r_sorted = rev.sort_values("total_revenue")
colors_bar = [PALETTE[i % len(PALETTE)] for i in range(len(r_sorted))]
bars = ax_rev.barh(r_sorted["short"], r_sorted["total_revenue"],
                   color=colors_bar, edgecolor="white", linewidth=0.5)
for bar, val in zip(bars, r_sorted["total_revenue"]):
    ax_rev.text(val + 50, bar.get_y() + bar.get_height()/2,
                f"${val:,.0f}", va="center", fontsize=8)
ax_rev.set_title("Tổng Doanh thu / Chi nhánh")
ax_rev.set_xlabel("USD (monthly)")
ax_rev.set_xlim(0, r_sorted["total_revenue"].max() * 1.2)
 
# ── 1C. Growth Score ranking ──────────────────────────────────────────────────
ax_gs = fig1.add_subplot(gs[1, 2])
g_sorted = rev.sort_values("growth_score")
cmap_gs = plt.cm.RdYlGn
norm_gs = plt.Normalize(g_sorted["growth_score"].min(), g_sorted["growth_score"].max())
bar_colors = [cmap_gs(norm_gs(v)) for v in g_sorted["growth_score"]]
bars2 = ax_gs.barh(g_sorted["short"], g_sorted["growth_score"],
                   color=bar_colors, edgecolor="white", linewidth=0.5)
for bar, val in zip(bars2, g_sorted["growth_score"]):
    ax_gs.text(val + 0.2, bar.get_y() + bar.get_height()/2,
               f"{val:.1f}", va="center", fontsize=8)
ax_gs.axvline(g_sorted["growth_score"].mean(), color=C["red"], lw=1.5, linestyle="--",
              label=f"Avg {g_sorted['growth_score'].mean():.1f}")
ax_gs.set_title("Growth Score (mở rộng)")
ax_gs.set_xlabel("Score (0-100)")
ax_gs.legend(fontsize=8)
ax_gs.set_xlim(0, 100)
 
# ── 1D. Revenue vs Members scatter ────────────────────────────────────────────
ax_sc = fig1.add_subplot(gs[2, :])
for i, (city, row) in enumerate(rev.iterrows()):
    ax_sc.scatter(row["members"], row["total_revenue"],
                  s=row["growth_score"]*12, color=PALETTE[i],
                  alpha=0.85, zorder=3, edgecolors="white", linewidths=1)
    ax_sc.annotate(row["short"],
                   xy=(row["members"], row["total_revenue"]),
                   xytext=(row["members"] + 2, row["total_revenue"] + 30),
                   fontsize=8.5, color=C["title"], fontweight="bold",
                   arrowprops=dict(arrowstyle="-", color=C["grid"], lw=0.8))
 
# Quadrant lines
mx, my = rev["members"].mean(), rev["total_revenue"].mean()
ax_sc.axvline(mx, color=C["axis"], lw=1, linestyle="--", alpha=0.5)
ax_sc.axhline(my, color=C["axis"], lw=1, linestyle="--", alpha=0.5)
 
# Quadrant labels
xlim = ax_sc.get_xlim(); ylim = ax_sc.get_ylim()
quad_labels = [
    (mx*0.7, my*1.08, "🌟 Ít thành viên\nDoanh thu cao\n→ Tiềm năng tăng trưởng"),
    (mx*1.15, my*1.08, "👑 Nhiều thành viên\nDoanh thu cao\n→ Chi nhánh đầu tàu"),
    (mx*0.7, my*0.88, "⚠️ Ít thành viên\nDoanh thu thấp\n→ Cần kích hoạt"),
    (mx*1.15, my*0.88, "📈 Nhiều thành viên\nDoanh thu thấp\n→ Upsell cần thiết"),
]
for qx, qy, ql in quad_labels:
    ax_sc.text(qx, qy, ql, fontsize=7.5, color=C["axis"], alpha=0.7,
               ha="center", va="center",
               bbox=dict(boxstyle="round", facecolor=C["bg"], edgecolor=C["grid"],
                         alpha=0.5, linewidth=0.5))
 
ax_sc.set_title("Ma trận Thành viên × Doanh thu (size = Growth Score)")
ax_sc.set_xlabel("Số thành viên"); ax_sc.set_ylabel("Tổng doanh thu (USD/tháng)")
 
fig1.savefig("../Output/05_location_revenue_map.png",
             dpi=150, bbox_inches="tight")
print("✔ Figure 1 saved: 05_location_revenue_map.png")
plt.close(fig1)

✔ Figure 1 saved: 05_location_revenue_map.png


### MULTI-LOCATION BEHAVIOR & MEMBERSHIP MIX

In [11]:
fig2 = plt.figure(figsize=(22, 16), facecolor=C["bg"])
fig2.suptitle("HÀNH VI THÀNH VIÊN THEO CHI NHÁNH & KHOẢNG CÁCH",
              fontsize=17, fontweight="bold", color=C["title"], y=0.98)
 
gs2 = gridspec.GridSpec(3, 3, figure=fig2, hspace=0.5, wspace=0.38,
                        left=0.06, right=0.97, top=0.93, bottom=0.04)
 
# ── 2A. Visit/week by city ────────────────────────────────────────────────────
ax = fig2.add_subplot(gs2[0, 0])
v_sorted = rev.sort_values("avg_visit", ascending=True)
colors_v = [C["green"] if v >= rev["avg_visit"].mean() else C["amber"]
            for v in v_sorted["avg_visit"]]
bars = ax.barh(v_sorted["short"], v_sorted["avg_visit"],
               color=colors_v, edgecolor="white", linewidth=0.5)
ax.axvline(rev["avg_visit"].mean(), color=C["red"], lw=1.5, linestyle="--",
           label=f"Avg {rev['avg_visit'].mean():.2f}")
for bar, val in zip(bars, v_sorted["avg_visit"]):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f"{val:.2f}", va="center", fontsize=8)
ax.set_title("Tần suất tập TB (lần/tuần)")
ax.set_xlabel("Lần / tuần")
ax.legend(fontsize=8)
 
# ── 2B. Duration by city ──────────────────────────────────────────────────────
ax = fig2.add_subplot(gs2[0, 1])
d_sorted = rev.sort_values("avg_duration", ascending=True)
colors_d = [C["blue"] if v >= rev["avg_duration"].mean() else C["amber"]
            for v in d_sorted["avg_duration"]]
bars = ax.barh(d_sorted["short"], d_sorted["avg_duration"],
               color=colors_d, edgecolor="white", linewidth=0.5)
ax.axvline(rev["avg_duration"].mean(), color=C["red"], lw=1.5, linestyle="--",
           label=f"Avg {rev['avg_duration'].mean():.0f}")
for bar, val in zip(bars, d_sorted["avg_duration"]):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f"{val:.0f}m", va="center", fontsize=8)
ax.set_title("Thời gian tập TB (phút)")
ax.set_xlabel("Phút")
ax.legend(fontsize=8)
 
# ── 2C. Avg Price by city ─────────────────────────────────────────────────────
ax = fig2.add_subplot(gs2[0, 2])
p_sorted = rev.sort_values("avg_price", ascending=True)
colors_p = [C["purple"] if v >= rev["avg_price"].mean() else C["axis"]
            for v in p_sorted["avg_price"]]
bars = ax.barh(p_sorted["short"], p_sorted["avg_price"],
               color=colors_p, edgecolor="white", linewidth=0.5)
ax.axvline(rev["avg_price"].mean(), color=C["red"], lw=1.5, linestyle="--",
           label=f"Avg ${rev['avg_price'].mean():.1f}")
for bar, val in zip(bars, p_sorted["avg_price"]):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2,
            f"${val:.1f}", va="center", fontsize=8)
ax.set_title("Giá TB / thành viên (USD/tháng)")
ax.set_xlabel("USD")
ax.legend(fontsize=8)
 
# ── 2D. Membership mix stacked bar ────────────────────────────────────────────
ax = fig2.add_subplot(gs2[1, :2])
mem_mix = df.groupby(["home_gym_location","membership_type"]).size().unstack(fill_value=0)
mem_mix_pct = mem_mix.div(mem_mix.sum(axis=1), axis=0) * 100
mem_mix_pct.index = [SHORT[c] for c in mem_mix_pct.index]
mem_order = mem_mix_pct.sum(axis=1).sort_values(ascending=False).index
mem_mix_pct = mem_mix_pct.loc[mem_order]
 
bottom = np.zeros(len(mem_mix_pct))
mem_colors = {"Elite": C["purple"], "Premium": C["blue"],
              "Standard": C["teal"], "Basic": C["amber"]}
for mtype in ["Elite","Premium","Standard","Basic"]:
    if mtype in mem_mix_pct.columns:
        vals = mem_mix_pct[mtype].values
        bars = ax.bar(mem_mix_pct.index, vals, bottom=bottom,
                      label=mtype, color=mem_colors.get(mtype, C["axis"]),
                      edgecolor="white", linewidth=0.5)
        for bar, val, bot in zip(bars, vals, bottom):
            if val > 6:
                ax.text(bar.get_x() + bar.get_width()/2,
                        bot + val/2, f"{val:.0f}%",
                        ha="center", va="center", fontsize=7.5,
                        color="white", fontweight="bold")
        bottom += vals
 
ax.set_title("Tỷ lệ Membership Mix theo Chi nhánh")
ax.set_ylabel("% thành viên")
ax.set_ylim(0, 110)
ax.legend(title="Membership", fontsize=8, loc="upper right")
ax.tick_params(axis="x", rotation=20)
 
# ── 2E. Multi-location % vs Visit rate ───────────────────────────────────────
ax = fig2.add_subplot(gs2[1, 2])
for i, (city, row) in enumerate(rev.iterrows()):
    ax.scatter(row["pct_multi"]*100, row["avg_visit"],
               s=row["members"]*2, color=PALETTE[i],
               alpha=0.8, edgecolors="white", linewidths=1, zorder=3)
    ax.annotate(row["short"], xy=(row["pct_multi"]*100, row["avg_visit"]),
                xytext=(1, 0.01), textcoords="offset points",
                fontsize=7.5, color=C["title"])
 
from scipy import stats as sp_stats
x_ml = rev["pct_multi"]*100
y_vw = rev["avg_visit"]
m, b, r, p, _ = sp_stats.linregress(x_ml, y_vw)
xr = np.linspace(x_ml.min(), x_ml.max(), 100)
ax.plot(xr, m*xr+b, color=C["red"], lw=1.5, linestyle="--", label=f"r={r:.2f}")
ax.set_title("Multi-location Access vs Tần suất tập\n(size = số thành viên)")
ax.set_xlabel("% có Multi-location Access")
ax.set_ylabel("Lần/tuần TB")
ax.legend(fontsize=8)
 
# ── 2F. Coverage gap heatmap ──────────────────────────────────────────────────
ax = fig2.add_subplot(gs2[2, :])
dm_short = dist_mat.copy()
dm_short.index = [SHORT[c] for c in dm_short.index]
dm_short.columns = [SHORT[c] for c in dm_short.columns]
 
# Mask upper triangle + diagonal
mask = np.triu(np.ones_like(dm_short, dtype=bool))
sns.heatmap(dm_short, mask=mask, ax=ax, annot=True, fmt=".0f",
            cmap="YlOrRd_r", vmin=0, vmax=800,
            linewidths=0.5, linecolor=C["bg"],
            annot_kws={"size": 8}, cbar_kws={"shrink": 0.5})
ax.set_title("Ma trận Khoảng cách giữa các Chi nhánh (km)\n→ Ô màu đỏ đậm = khoảng trống lớn, tiềm năng mở chi nhánh trung gian")
ax.tick_params(axis="x", rotation=40)
ax.tick_params(axis="y", rotation=0)
 
fig2.savefig("../Output/06_behavior_by_location.png",
             dpi=150, bbox_inches="tight")
print("✔ Figure 2 saved: 06_behavior_by_location.png")
plt.close(fig2)

✔ Figure 2 saved: 06_behavior_by_location.png


### EXPANSION STRATEGY DASHBOARD

In [12]:
fig3 = plt.figure(figsize=(22, 16), facecolor=C["bg"])
fig3.suptitle("CHIẾN LƯỢC MỞ RỘNG CHI NHÁNH — PHÂN TÍCH & ĐỀ XUẤT",
              fontsize=17, fontweight="bold", color=C["title"], y=0.98)
 
gs3 = gridspec.GridSpec(3, 3, figure=fig3, hspace=0.52, wspace=0.38,
                        left=0.06, right=0.97, top=0.93, bottom=0.04)
 
# ── 3A. 2x2 Priority Matrix: Revenue × Growth Score ───────────────────────────
ax = fig3.add_subplot(gs3[:2, :2])
 
rev_vals = rev["total_revenue"].values
gs_vals  = rev["growth_score"].values
rev_mean = rev_vals.mean()
gs_mean  = gs_vals.mean()
 
for i, (city, row) in enumerate(rev.iterrows()):
    rv = row["total_revenue"]
    gs_v = row["growth_score"]
    size = row["members"] * 3
 
    color = (C["green"]  if rv >= rev_mean and gs_v >= gs_mean else
             C["blue"]   if rv < rev_mean  and gs_v >= gs_mean else
             C["amber"]  if rv >= rev_mean and gs_v < gs_mean  else C["axis"])
 
    ax.scatter(rv, gs_v, s=size, color=color, alpha=0.85,
               edgecolors="white", linewidths=1.5, zorder=3)
    ax.annotate(row["short"], xy=(rv, gs_v),
                xytext=(50, 5), textcoords="offset points",
                fontsize=9, fontweight="bold", color=C["title"],
                bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                          edgecolor=C["grid"], alpha=0.85))
 
ax.axvline(rev_mean, color=C["axis"], lw=1.5, linestyle="--", alpha=0.6)
ax.axhline(gs_mean,  color=C["axis"], lw=1.5, linestyle="--", alpha=0.6)
 
quad_texts = [
    (rev_mean * 0.72, gs_mean * 1.05, "🚀 TIỀM NĂNG MỞ RỘNG\nDoanh thu thấp nhưng\nGrowth Score cao", C["blue"]),
    (rev_mean * 1.15, gs_mean * 1.05, "👑 NHÂN RỘNG MÔ HÌNH\nDoanh thu cao &\nGrowth Score cao", C["green"]),
    (rev_mean * 0.72, gs_mean * 0.88, "⚠️ CẦN TỐI ƯU HOÁ\nDoanh thu thấp &\nGrowth Score thấp", C["amber"]),
    (rev_mean * 1.15, gs_mean * 0.88, "📊 KHAI THÁC TỐT HƠN\nDoanh thu cao nhưng\nGrowth Score thấp", C["rose"]),
]
for qx, qy, ql, qc in quad_texts:
    ax.text(qx, qy, ql, fontsize=8.5, color=qc, ha="center", va="center",
            bbox=dict(boxstyle="round", facecolor=C["bg"],
                      edgecolor=qc, alpha=0.6, linewidth=1))
 
legend_els = [
    mpatches.Patch(facecolor=C["green"],  label="Doanh thu cao + Growth cao → Nhân rộng"),
    mpatches.Patch(facecolor=C["blue"],   label="Doanh thu thấp + Growth cao → Tiềm năng"),
    mpatches.Patch(facecolor=C["amber"],  label="Doanh thu cao + Growth thấp → Khai thác"),
    mpatches.Patch(facecolor=C["axis"],   label="Doanh thu thấp + Growth thấp → Tối ưu"),
]
ax.legend(handles=legend_els, fontsize=8, loc="lower right", framealpha=0.85)
ax.set_title("Ma trận Ưu tiên Mở rộng: Doanh thu × Growth Score\n(size = số thành viên)")
ax.set_xlabel("Tổng doanh thu tháng (USD)")
ax.set_ylabel("Growth Score (0-100)")
 
# ── 3B. Isolation score (nearest km) vs revenue ───────────────────────────────
ax = fig3.add_subplot(gs3[0, 2])
for i, (city, row) in enumerate(rev.iterrows()):
    ax.scatter(row["nearest_km"], row["total_revenue"],
               s=100, color=PALETTE[i], alpha=0.85, edgecolors="white", linewidths=1)
    ax.annotate(row["short"], xy=(row["nearest_km"], row["total_revenue"]),
                xytext=(2, 2), textcoords="offset points", fontsize=7.5)
ax.set_title("Mức độ cô lập (nearest km)\nvs Doanh thu")
ax.set_xlabel("Khoảng cách đến gym gần nhất (km)")
ax.set_ylabel("Tổng doanh thu (USD)")
 
# ── 3C. Elite+Premium % vs Avg Price ─────────────────────────────────────────
ax = fig3.add_subplot(gs3[1, 2])
rev["pct_high_tier"] = rev["pct_premium"] + rev["pct_elite"]
for i, (city, row) in enumerate(rev.iterrows()):
    ax.scatter(row["pct_high_tier"], row["avg_price"],
               s=row["members"]*2, color=PALETTE[i], alpha=0.85,
               edgecolors="white", linewidths=1)
    ax.annotate(row["short"], xy=(row["pct_high_tier"], row["avg_price"]),
                xytext=(1, 1), textcoords="offset points", fontsize=7.5)
ax.set_title("% Premium+Elite vs Giá TB\n(size = số thành viên)")
ax.set_xlabel("% Premium + Elite")
ax.set_ylabel("Avg Final Price (USD)")
 
# ── 3D. Expansion recommendation cards ───────────────────────────────────────
ax = fig3.add_subplot(gs3[2, :])
ax.axis("off")
 
recommendations = [
    {
        "rank": "#1",
        "location": "Riverside / San Bernardino",
        "type": "Khoảng trống địa lý",
        "reason": "Nằm giữa Anaheim (27km) và San Diego (143km). Dân số 3.2M, không có chi nhánh nào. Multi-location members từ Anaheim/SD có thể là base khách hàng.",
        "signal": "★★★★★",
        "color": C["green"],
    },
    {
        "rank": "#2",
        "location": "San Jose → Mở thêm điểm nội thị",
        "type": "High-value market cần upsell",
        "reason": "San Jose có avg_price cao nhất ($37.2), growth_score #1 (80.3), nhưng chỉ 167 thành viên — ít nhất chuỗi. Demand rõ ràng nhưng capacity đang bị giới hạn.",
        "signal": "★★★★☆",
        "color": C["blue"],
    },
    {
        "rank": "#3",
        "location": "Modesto / Stockton",
        "type": "Khoảng trống Fresno–Sacramento 254km",
        "reason": "Hành lang 254km giữa Fresno và Sacramento không có chi nhánh. Fresno có doanh thu cao nhất ($8,983). Modesto dân số 220K là bước đệm tự nhiên.",
        "signal": "★★★★☆",
        "color": C["purple"],
    },
    {
        "rank": "#4",
        "location": "Los Angeles → Thêm chi nhánh nội đô",
        "type": "Consolidate & Scale",
        "reason": "LA có visit/week cao nhất (2.87), avg_price tốt ($36.9), growth_score #2 (80.1). Nhưng chỉ 173 thành viên — cần thêm location để capture demand trong thành phố lớn nhất CA.",
        "signal": "★★★☆☆",
        "color": C["amber"],
    },
]
 
card_w = 0.23
for i, rec in enumerate(recommendations):
    x0 = 0.01 + i * 0.25
    # Card background
    ax.add_patch(FancyBboxPatch((x0, 0.05), card_w, 0.9,
                                boxstyle="round,pad=0.02",
                                linewidth=2, edgecolor=rec["color"],
                                facecolor=rec["color"] + "11",
                                transform=ax.transAxes, zorder=1))
    # Header strip
    ax.add_patch(FancyBboxPatch((x0, 0.78), card_w, 0.17,
                                boxstyle="round,pad=0.01",
                                linewidth=0, facecolor=rec["color"],
                                transform=ax.transAxes, zorder=2))
 
    ax.text(x0 + card_w/2, 0.88, rec["rank"],
            transform=ax.transAxes, ha="center", va="center",
            fontsize=18, fontweight="bold", color="white", zorder=3)
 
    ax.text(x0 + card_w/2, 0.73, rec["location"],
            transform=ax.transAxes, ha="center", va="center",
            fontsize=9.5, fontweight="bold", color=rec["color"], zorder=3)
 
    ax.text(x0 + card_w/2, 0.63, f"[{rec['type']}]",
            transform=ax.transAxes, ha="center", va="center",
            fontsize=8, color=C["axis"], style="italic", zorder=3)
 
    ax.text(x0 + card_w/2, 0.39, rec["reason"],
            transform=ax.transAxes, ha="center", va="center",
            fontsize=8, color=C["title"], wrap=True, zorder=3,
            multialignment="center")
 
    ax.text(x0 + card_w/2, 0.15, f"Độ ưu tiên: {rec['signal']}",
            transform=ax.transAxes, ha="center", va="center",
            fontsize=10, color=rec["color"], zorder=3)
 
ax.set_title("ĐỀ XUẤT MỞ RỘNG CHI NHÁNH — Xếp hạng theo mức độ ưu tiên",
             fontsize=12, pad=12)
 
fig3.savefig("../Output/07_expansion_strategy.png",
             dpi=150, bbox_inches="tight")
print("✔ Figure 3 saved: 07_expansion_strategy.png")
plt.close(fig3)
 
# ── Console summary ───────────────────────────────────────────────────────────
print()
print("="*65)
print("  LOCATION INSIGHTS SUMMARY")
print("="*65)
print(f"\n{'City':<22} {'Members':>8} {'Revenue':>10} {'Avg$':>7} {'Visit':>6} {'Growth':>7}")
print("-"*65)
for city, row in rev.sort_values("growth_score", ascending=False).iterrows():
    print(f"{row['short']:<22} {row['members']:>8} ${row['total_revenue']:>9,.0f} "
          f"${row['avg_price']:>6.1f} {row['avg_visit']:>6.2f} {row['growth_score']:>7.1f}")
 
print()
print("KHOẢNG CÁCH LỚN NHẤT (tiềm năng mở mới):")
for c in cities:
    d_near = dist_mat.loc[c].drop(c).min()
    nc = dist_mat.loc[c].drop(c).idxmin()
    if d_near > 100:
        print(f"  {SHORT[c]:20s} ↔ {SHORT[nc]:20s}  {d_near:.0f} km")
print()
print("✔ Tất cả biểu đồ đã lưu trong ../Output/")

✔ Figure 3 saved: 07_expansion_strategy.png

  LOCATION INSIGHTS SUMMARY

City                    Members    Revenue    Avg$  Visit  Growth
-----------------------------------------------------------------
Bakersfield                 189 $    6,842 $  36.2   2.54    81.0
San Diego                   201 $    7,395 $  36.8   2.70    80.5
San Jose                    167 $    6,206 $  37.2   2.80    77.1
Los Angeles                 173 $    6,386 $  36.9   2.87    74.5
Fresno                      265 $    8,983 $  33.9   2.61    74.1
Sacramento                  226 $    7,359 $  32.6   2.59    70.5
Anaheim                     188 $    6,023 $  32.0   2.88    68.4
Oakland                     165 $    5,479 $  33.2   2.62    67.3
San Francisco               201 $    6,798 $  33.8   2.65    65.5
Long Beach                  223 $    7,402 $  33.2   2.58    63.7

KHOẢNG CÁCH LỚN NHẤT (tiềm năng mở mới):
  Bakersfield          ↔ Los Angeles           163 km
  Fresno               ↔ Bakersfield  

# -------------- Nhận xét --------------------

### 🗺️ Insights Doanh thu theo Vị trí

| Chi nhánh     | Doanh thu/tháng | Avg Price | Visit/week | Growth Score |
|---------------|-----------------|-----------|------------|--------------|
| Fresno        | $8,983 🥇       | $33.9     | 2.61       | 74.1         |
| Long Beach    | $7,402          | $33.2     | 2.58       | 63.7         |
| San Diego     | $7,395          | $36.8     | 2.70       | 80.5         |
| San Jose      | $6,206          | $37.2 🥇  | 2.80       | 77.1         |

---

#### 🔍 Insight quan trọng

> **Nghịch lý doanh thu:**
- **Fresno** đạt doanh thu cao nhất nhờ **số lượng thành viên lớn (265)**  
- Trong khi đó, **San Jose** và **Los Angeles** có **unit revenue cao hơn**  
→ *Ít thành viên hơn nhưng mỗi người chi tiêu nhiều hơn*

---

### 📍 Phân tích Khoảng cách & Hành vi

- Dataset sử dụng **tọa độ gym (không phải nhà riêng khách hàng)**  
→ Phân tích tập trung vào **khoảng cách giữa các chi nhánh**

##### 🔗 Key Findings

- **Multi-location access** có tương quan dương với thời gian tập  
  → Thành viên sử dụng nhiều chi nhánh tập **lâu hơn ~4.2 phút**

- **Anaheim & Los Angeles**
  - Tần suất tập cao nhất: **2.87 – 2.88 lần/tuần**
  - Không phải top doanh thu  
  → Nhóm khách hàng **trung thành & gắn bó cao**

---

### 🚀 Đề xuất Mở rộng (ưu tiên từ cao → thấp)

#### ⭐⭐⭐⭐⭐ #1 — Riverside / San Bernardino
- Khoảng trống địa lý **143km** giữa Anaheim và San Diego  
- Dân số ~**3.2 triệu**
- Không có chi nhánh nào trong mạng hiện tại  

👉 **Cơ hội mở rộng mạnh nhất (blue ocean)**

---

#### ⭐⭐⭐⭐ #2 — San Jose (mở thêm chi nhánh nội đô)
- **Growth Score cao (77.1)**
- **Avg Price cao nhất ($37.2)**
- Chỉ **167 thành viên**

👉 **Cầu > Cung → mở rộng để tận dụng willingness-to-pay**

---

#### ⭐⭐⭐⭐ #3 — Modesto / Stockton
- Khoảng trống **254km** giữa Fresno và Sacramento  
- Nằm trong hành lang Central Valley  

👉 **Mở rộng để “nối thị trường” + tăng coverage**

---

#### ⭐⭐⭐ #4 — Los Angeles (mở thêm điểm)
- **Visit rate cao nhất (~2.87)**
- Avg price tốt  
- Chỉ **173 thành viên**

👉 **Thị trường lớn nhưng chưa khai thác hết**

---

### 📌 Kết luận chiến lược

- **Doanh thu ≠ giá trị khách hàng**
- Cần phân biệt:
  - **Volume-driven (Fresno)**  
  - **Value-driven (San Jose, LA)**  

👉 Chiến lược tối ưu:
- Mở rộng ở **vùng trống địa lý**
- Scale ở **thị trường có willingness-to-pay cao**